# YOLO: быстрый старт для инференса

Этот ноутбук показывает минимальный и при этом удобный старт для работы с YOLO. Здесь много комментариев, чтобы можно было быстро адаптировать шаблон под своё задание.


## Что делает этот ноутбук

- проверяет окружение
- показывает базовый импорт
- загружает предобученную YOLO-модель
- запускает инференс на одном изображении
- запускает инференс на папке изображений
- показывает, как сохранить результаты
- объясняет, какие параметры чаще всего приходится менять


In [ ]:

# Базовые импорты.
# Здесь только самые нужные библиотеки для быстрого старта.
from pathlib import Path
import os
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

from ultralytics import YOLO

print("Python:", sys.version)
print("Текущая папка:", Path.cwd())


In [ ]:

# При желании можно менять модель здесь.
# Самый быстрый и лёгкий вариант для старта — yolov8n.pt.
# Более тяжёлые модели обычно точнее, но медленнее.
MODEL_NAME = "yolov8n.pt"

# Путь к изображению для одиночного теста.
# Замените на свой файл при реальной работе.
IMAGE_PATH = "sample.jpg"

# Путь к папке с изображениями.
# Если хотите прогнать сразу набор файлов, используйте папку.
IMAGE_DIR = "images"

# Папка для сохранения результатов.
OUTPUT_DIR = "runs_inference"


## Загрузка модели

YOLO из пакета `ultralytics` позволяет загрузить модель одной строкой. При первом запуске веса будут скачаны автоматически.


In [ ]:

# Загружаем модель.
# Если файл весов уже есть локально, можно передать путь к нему.
model = YOLO(MODEL_NAME)
model


## Инференс на одном изображении

Ниже пример для одиночного файла. Код обёрнут в проверку существования файла, чтобы ноутбук не падал на пустой заготовке.


In [ ]:

image_path = Path(IMAGE_PATH)

if image_path.exists():
    # predict возвращает список результатов.
    # save=True сохраняет визуализации в runs/...
    results = model.predict(
        source=str(image_path),
        conf=0.25,          # порог confidence
        imgsz=640,          # размер изображения для модели
        save=True,
        project=OUTPUT_DIR,
        name="single_image",
        exist_ok=True,
        verbose=False,
    )

    print("Количество объектов results:", len(results))
    print("Инференс выполнен успешно")
else:
    print(f"Файл {image_path} не найден. Укажите своё изображение и перезапустите ячейку.")


## Просмотр предсказаний

В объекте `results[0]` есть боксы, классы, confidence и готовая отрисованная картинка.


In [ ]:

if 'results' in locals() and len(results) > 0:
    r = results[0]

    # Таблица по найденным объектам.
    boxes = r.boxes
    if boxes is not None and len(boxes) > 0:
        classes = boxes.cls.cpu().numpy().astype(int)
        confs = boxes.conf.cpu().numpy()
        xyxy = boxes.xyxy.cpu().numpy()

        rows = []
        for cls_id, conf, coords in zip(classes, confs, xyxy):
            rows.append({
                "class_id": cls_id,
                "class_name": model.names.get(int(cls_id), str(cls_id)),
                "confidence": float(conf),
                "x1": float(coords[0]),
                "y1": float(coords[1]),
                "x2": float(coords[2]),
                "y2": float(coords[3]),
            })

        df_pred = pd.DataFrame(rows)
        display(df_pred)
    else:
        print("Объекты не найдены")
else:
    print("Сначала выполните инференс")


In [ ]:

if 'results' in locals() and len(results) > 0:
    # plot() возвращает numpy-массив с отрисованными bbox.
    plotted = results[0].plot()

    plt.figure(figsize=(10, 8))
    plt.imshow(plotted)
    plt.axis("off")
    plt.title("YOLO prediction")
    plt.show()


## Инференс на папке изображений

Полезно, если нужно быстро прогнать целый каталог без написания отдельного скрипта.


In [ ]:

image_dir = Path(IMAGE_DIR)

if image_dir.exists() and image_dir.is_dir():
    batch_results = model.predict(
        source=str(image_dir),
        conf=0.25,
        imgsz=640,
        save=True,
        project=OUTPUT_DIR,
        name="batch_images",
        exist_ok=True,
        verbose=False,
    )

    print("Файлов обработано:", len(batch_results))
else:
    print(f"Папка {image_dir} не найдена. Создайте её и положите туда изображения.")


## Полезные параметры `predict`

Чаще всего на практике меняют:

- `conf` — порог confidence
- `imgsz` — размер входного изображения
- `save` — сохранять ли отрисованные результаты
- `device` — например `0` для GPU или `cpu`
- `classes` — ограничить детекцию по конкретным классам
- `max_det` — ограничить число найденных объектов


In [ ]:

# Пример инференса с более явными параметрами.
# Это заготовка: снимите комментарии и используйте при необходимости.

# tuned_results = model.predict(
#     source="images",
#     conf=0.35,
#     imgsz=960,
#     device="cpu",   # или 0, если есть CUDA GPU
#     max_det=100,
#     save=True,
#     project=OUTPUT_DIR,
#     name="tuned_run",
#     exist_ok=True,
# )


## Что делать дальше

Если нужно обучать модель на своих данных, переходите ко второму ноутбуку. Там показана подготовка структуры датасета, генерация `dataset.yaml` и запуск обучения.
